# 02 — Preprocessing
Lemmatisation + suppression stopwords avec spaCy. Produit un corpus tokenisé par année.

In [7]:
import pandas as pd
import spacy
import pickle
from pathlib import Path
from tqdm import tqdm

nlp = spacy.load('fr_core_news_lg', disable=['parser', 'ner'])
nlp.max_length = 3_000_000

df = pd.read_csv('../data/metadata/corpus_complet.csv')
print(f'Corpus chargé : {len(df)} documents, années {sorted(df.year.unique())}')

Corpus chargé : 21697 documents, années [np.int64(1973), np.int64(1978), np.int64(1981), np.int64(1988), np.int64(1993)]


In [8]:
import re

def clean_ocr(text):
    # 1. Mots coupés en fin de ligne : "tra-\nvail" -> "travail"
    text = re.sub(r'(\w+)-\n(\w+)', r'\1\2', text)
    # 2. Mots coupés avec espace après le tiret (cas dominant : ~3400 occurrences)
    #    "tra- vail", "Hexa- gone", "fisca- lité" -> recoller.
    #    On exige PAS d'espace AVANT le tiret pour épargner "trois - quatre".
    text = re.sub(r'(\w{2,})-\s+(\w{2,})', r'\1\2', text)
    # 3. Sauts de ligne simples -> espace
    text = re.sub(r'(?<!\n)\n(?!\n)', ' ', text)
    # 4. Espaces multiples
    text = re.sub(r' +', ' ', text)
    return text.strip()

sample = df.iloc[0]['text']
print('AVANT:')
print(sample[:400])
print('\nAPRÈS:')
print(clean_ocr(sample)[:400])

n_before = sum(len(re.findall(r'\w-\s+\w', str(t))) for t in df['text'].head(500))
n_after  = sum(len(re.findall(r'\w-\s+\w', clean_ocr(str(t)))) for t in df['text'].head(500))
print(f'\nCésures "mot- mot" sur 500 docs : {n_before} avant -> {n_after} après')

AVANT:
10me Circonscription
Elections Législatives - 4 Mars 1973
POURQUOI UNE OPPOSITION GAULLISTE ?
La "majorité" a trompé les Français : élue, certes par la grande peur de 1968, mais aussi au nom du gaullisme, elle en a trahi les principes essentiels.
*
- L'Etat livré aux puissances d'argent et compromis par les scandales.
- Les plus pauvres privés du fruit de l'expansion.
- Les travailleurs exploités,

APRÈS:
10me Circonscription Elections Législatives - 4 Mars 1973 POURQUOI UNE OPPOSITION GAULLISTE ? La "majorité" a trompé les Français : élue, certes par la grande peur de 1968, mais aussi au nom du gaullisme, elle en a trahi les principes essentiels. * - L'Etat livré aux puissances d'argent et compromis par les scandales. - Les plus pauvres privés du fruit de l'expansion. - Les travailleurs exploités,

Césures "mot- mot" sur 500 docs : 3443 avant -> 13 après


In [9]:
EXTRA_STOPS = {
    'monsieur','madame','cher','chère','électeur','électrice','candidat',
    'circonscription','département','assemblée','nationale','république',
    'française','scrutin','suffrage','liste','bulletin','vote','voix',
    'premier','second','tour','élection','législatif','législative',
    'permettre','faire','vouloir','devoir','pouvoir','aller','mettre',
    'rendre','prendre','donner','avoir','être','falloir','savoir','voir',
    'venir','tenir','partir','sciences','cevipof','fonds','mars', 'suppléant',
    'science', 'juin', 'novembre', 'etat'
}

def is_valid_token(token):
    lemma = token.lemma_.lower()
    return (
        token.is_alpha
        and not token.is_stop
        and len(lemma) > 3
        and not token.like_num
        and token.has_vector
        and token.vector_norm > 0
        and lemma not in EXTRA_STOPS
    )

def preprocess(text):
    text = clean_ocr(text)          
    doc = nlp(str(text)[:50000])
    return [token.lemma_.lower() for token in doc if is_valid_token(token)]
# Test rapide
sample = df.iloc[0]['text']
result = preprocess(sample)
print(f'Exemple — texte brut ({len(sample.split())} mots) → lemmes ({len(result)} tokens)')
print('Premiers lemmes:', result[:20])

Exemple — texte brut (789 mots) → lemmes (309 tokens)
Premiers lemmes: ['election', 'opposition', 'majorité', 'tromper', 'français', 'élire', 'grand', 'peur', 'gaullisme', 'trahir', 'principe', 'essentiel', 'livrer', 'puissance', 'argent', 'compromettre', 'scandale', 'pauvre', 'priver', 'fruit']


In [ ]:
# Preprocessing par année
Path('../data/processed').mkdir(exist_ok=True)
corpus_by_year = {}
doc_ids_by_year = {}    # même ordre que corpus_by_year, pour merge avec metadata

for year in sorted(df['year'].unique()):
    sub = df[df['year'] == year]
    texts = sub['text'].fillna('').tolist()
    ids   = sub['document_id'].tolist()
    print(f'\n=== {year} : {len(texts)} documents ===')

    tokenized, kept_ids = [], []
    for doc_id, text in tqdm(list(zip(ids, texts)), desc=f'{year}'):
        tokens = preprocess(text)
        if len(tokens) > 10:
            tokenized.append(tokens)
            kept_ids.append(doc_id)

    corpus_by_year[year]  = tokenized
    doc_ids_by_year[year] = kept_ids

    with open(f'../data/processed/corpus_{year}.pkl', 'wb') as f:
        pickle.dump(tokenized, f)
    with open(f'../data/processed/doc_ids_{year}.pkl', 'wb') as f:
        pickle.dump(kept_ids, f)

    total_tokens = sum(len(t) for t in tokenized)
    print(f'   → {len(tokenized)} docs valides, {total_tokens:,} tokens après preprocessing')


=== 1973 : 3921 documents ===


1973: 100%|██████████| 3921/3921 [02:58<00:00, 21.92it/s]


   → 3917 docs valides, 1,168,211 tokens après preprocessing

=== 1978 : 5030 documents ===


1978: 100%|██████████| 5030/5030 [03:47<00:00, 22.07it/s]


   → 5025 docs valides, 1,504,300 tokens après preprocessing

=== 1981 : 3182 documents ===


...

   → 3179 docs valides, 774,182 tokens après preprocessing

=== 1988 : 3628 documents ===


1988: 100%|██████████| 3628/3628 [01:52<00:00, 32.13it/s]


   → 3615 docs valides, 740,342 tokens après preprocessing

=== 1993 : 5936 documents ===


1993: 100%|██████████| 5936/5936 [03:38<00:00, 27.19it/s]


   → 5929 docs valides, 1,468,404 tokens après preprocessing

✅ Preprocessing terminé. Corpus + doc_ids sauvegardés dans data/processed/


In [11]:
# Normalisation des variantes d'accents (OCR perd souvent les accents)
# On regroupe "liberte"/"liberté", "securite"/"sécurité", "egalité"/"égalité", etc.
# Pour chaque forme désaccentuée, on garde la variante la plus fréquente du corpus.
import unicodedata
from collections import Counter, defaultdict

def strip_accents(s):
    return ''.join(c for c in unicodedata.normalize('NFD', s)
                   if unicodedata.category(c) != 'Mn')

# Fréquences globales (tous corpus confondus)
global_freq = Counter()
for tokens_list in corpus_by_year.values():
    for doc in tokens_list:
        global_freq.update(doc)

# Bucket par forme désaccentuée
buckets = defaultdict(list)
for tok, freq in global_freq.items():
    buckets[strip_accents(tok)].append((tok, freq))

# Map : variante minoritaire -> variante majoritaire
normalize_map = {}
for stripped, variants in buckets.items():
    if len(variants) > 1:
        best = max(variants, key=lambda x: x[1])[0]
        for tok, _ in variants:
            if tok != best:
                normalize_map[tok] = best

print(f'{len(normalize_map)} variantes accentuées seront fusionnées')
print('Exemples :')
for k, v in list(normalize_map.items())[:10]:
    print(f'  {k!r:20s} -> {v!r}  (freq cibles : {global_freq[k]} -> {global_freq[v]})')

# Appliquer la normalisation et re-sauvegarder
for year, docs in corpus_by_year.items():
    corpus_by_year[year] = [[normalize_map.get(t, t) for t in doc] for doc in docs]
    with open(f'../data/processed/corpus_{year}.pkl', 'wb') as f:
        pickle.dump(corpus_by_year[year], f)
print('\nCorpus sauvegardés avec accents normalisés.')

2093 variantes accentuées seront fusionnées
Exemples :
  'éléction'           -> 'election'  (freq cibles : 9 -> 12994)
  'majorite'           -> 'majorité'  (freq cibles : 3 -> 26018)
  'francais'           -> 'français'  (freq cibles : 57 -> 45544)
  'elire'              -> 'élire'  (freq cibles : 75 -> 7712)
  'exploite'           -> 'exploité'  (freq cibles : 4 -> 197)
  'fiscalite'          -> 'fiscalité'  (freq cibles : 71 -> 2457)
  'esperance'          -> 'espérance'  (freq cibles : 60 -> 1554)
  'necessaire'         -> 'nécessaire'  (freq cibles : 72 -> 5571)
  'independance'       -> 'indépendance'  (freq cibles : 294 -> 3994)
  'resistance'         -> 'résistance'  (freq cibles : 65 -> 1983)

Corpus sauvegardés avec accents normalisés.


In [ ]:
from collections import Counter

print(f'{"Année":<8} {"Docs":<8} {"Tokens":<12} {"Vocab":<10} {"Moy tokens/doc"}')
print('-' * 55)
for year in sorted(corpus_by_year.keys()):
    corp = corpus_by_year[year]
    all_tokens = [t for doc in corp for t in doc]
    vocab = len(set(all_tokens))
    avg = len(all_tokens) // len(corp) if corp else 0
    print(f'{year:<8} {len(corp):<8} {len(all_tokens):<12,} {vocab:<10,} {avg}')

Année    Docs     Tokens       Vocab      Moy tokens/doc
-------------------------------------------------------
1973     3917     1,168,211    19,211     298
1978     5025     1,504,300    22,211     299
1981     3179     774,182      17,158     243
1988     3615     740,342      17,153     204
1993     5929     1,468,404    23,100     247
